# PharmaTrade-MM Phase 1 Data Pipeline

This notebook reproduces the full Phase 1 processing flow:
- sentiment decay forward-fill
- FDA daily feature engineering
- unified dataset merge
- train/test split
- benchmark processing
- feature config export

Run cells top-to-bottom.


In [49]:
import json
import math
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd

TRAIN_START = "2018-02-20"
TRAIN_END = "2021-12-31"
TEST_START = "2022-01-03"
TEST_END = "2023-12-29"

MODEL_TICKERS = ["PFE", "JNJ", "MRK", "ABBV", "BMY", "AMGN", "GILD", "BIIB"]
EVENT_TYPES = ["ADCOM", "APPROVAL", "CRL"]
THERAPEUTIC_AREAS = [
    "Bone Disease",
    "Cardiology",
    "Dermatology",
    "Immunology",
    "Infectious Disease",
    "Neurology",
    "Oncology",
    "Psychiatry",
]

@dataclass
class Config:
    data_dir: Path
    out_dir: Path
    zip_path: Path
    extract_zip: bool
    sentiment_half_life: float


In [50]:
def ensure_data_dir(config: Config) -> Path:
    data_dir = config.data_dir
    if config.extract_zip and config.zip_path.exists():
        data_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(config.zip_path, "r") as zf:
            zf.extractall(data_dir)

    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir}")

    children = [p for p in data_dir.iterdir() if p.is_dir()]
    csv_count_here = len(list(data_dir.glob("*.csv")))
    if csv_count_here == 0 and len(children) == 1:
        nested = children[0]
        if len(list(nested.glob("*.csv"))) > 0:
            data_dir = nested

    return data_dir


def load_price(data_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(data_dir / "price_technicals.csv")
    df["date"] = pd.to_datetime(df["date"])
    df["ticker"] = df["ticker"].astype(str).str.upper()
    return df.sort_values(["ticker", "date"]).reset_index(drop=True)


def validate_price_alignment(price_df: pd.DataFrame) -> None:
    counts = price_df.groupby("ticker")["date"].nunique()
    min_cnt, max_cnt = int(counts.min()), int(counts.max())
    if min_cnt != max_cnt:
        print("WARNING: Date counts differ by ticker:")
        print(counts.to_string())
    else:
        print(f"Date alignment OK: each ticker has {min_cnt} rows.")


def load_sentiment(data_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(data_dir / "sentiment_daily.csv")
    df["date"] = pd.to_datetime(df["date"])
    df["ticker"] = df["ticker"].astype(str).str.upper()
    return df.sort_values(["ticker", "date"]).reset_index(drop=True)


def apply_sentiment_decay(sentiment_df: pd.DataFrame, half_life: float) -> pd.DataFrame:
    decay = math.exp(math.log(0.5) / half_life)
    out: List[pd.DataFrame] = []

    for ticker, grp in sentiment_df.groupby("ticker", sort=False):
        g = grp.sort_values("date").copy()
        last_pos = 0.0
        last_neg = 0.0
        dec_pos, dec_neg, dec_neu, dec_net = [], [], [], []

        for _, row in g.iterrows():
            filings = float(row.get("n_filings", 0) or 0)
            pos_raw = float(row.get("sentiment_pos", 0) or 0)
            neg_raw = float(row.get("sentiment_neg", 0) or 0)

            if filings > 0:
                last_pos = max(0.0, pos_raw)
                last_neg = max(0.0, neg_raw)
            else:
                last_pos *= decay
                last_neg *= decay

            neu = max(0.0, min(1.0, 1.0 - (last_pos + last_neg)))
            net = last_pos - last_neg

            dec_pos.append(last_pos)
            dec_neg.append(last_neg)
            dec_neu.append(neu)
            dec_net.append(net)

        g["sent_pos"] = dec_pos
        g["sent_neg"] = dec_neg
        g["sent_neu"] = dec_neu
        g["sent_net"] = dec_net
        out.append(g[["date", "ticker", "sent_pos", "sent_neg", "sent_neu", "sent_net", "n_filings"]])

    return pd.concat(out, ignore_index=True)


def load_fda_events(data_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(data_dir / "fda_event_calendar.csv")
    df["date"] = pd.to_datetime(df["date"])
    df["ticker"] = df["ticker"].astype(str).str.upper()
    df["event_type"] = df["event_type"].astype(str).str.upper()
    return df.sort_values(["ticker", "date"]).reset_index(drop=True)


In [51]:
def build_hist_reaction_map(fda_df: pd.DataFrame) -> Dict[Tuple[str, str], Tuple[float, float]]:
    train_end = pd.Timestamp(TRAIN_END)
    train_events = fda_df[fda_df["date"] <= train_end].copy()
    grouped = (
        train_events.groupby(["ticker", "event_type"], as_index=False)[["px_1d_ret_pct", "px_5d_ret_pct"]]
        .mean(numeric_only=True)
        .fillna(0.0)
    )

    reaction_map: Dict[Tuple[str, str], Tuple[float, float]] = {}
    for _, row in grouped.iterrows():
        reaction_map[(row["ticker"], row["event_type"])] = (
            float(row["px_1d_ret_pct"]),
            float(row["px_5d_ret_pct"]),
        )
    return reaction_map


def build_fda_daily_features(price_df: pd.DataFrame, fda_df: pd.DataFrame) -> pd.DataFrame:
    reaction_map = build_hist_reaction_map(fda_df)
    rows: List[dict] = []

    for ticker, pgrp in price_df.groupby("ticker", sort=False):
        events = fda_df[fda_df["ticker"] == ticker].sort_values("date").copy()
        event_dates = events["date"].tolist()

        for as_of in pgrp["date"].tolist():
            nearest = None
            dte = 999

            for idx, ed in enumerate(event_dates):
                delta = (ed - as_of).days
                if delta >= 0:
                    dte = min(delta, 90)
                    nearest = events.iloc[idx]
                    break

            event_type = None
            area = None
            hist1, hist5 = 0.0, 0.0
            is_window = 0

            if nearest is not None:
                event_type = str(nearest["event_type"]).upper()
                area = str(nearest["therapeutic_area"])
                hist1, hist5 = reaction_map.get((ticker, event_type), (0.0, 0.0))
                is_window = int(dte <= 5)

            rec = {
                "date": as_of,
                "ticker": ticker,
                "days_to_event": int(dte),
                "is_event_window": is_window,
                "hist_1d_ret": hist1,
                "hist_5d_ret": hist5,
            }

            for ev in EVENT_TYPES:
                rec[f"evt_{ev}"] = int(event_type == ev)

            for ta in THERAPEUTIC_AREAS:
                rec[f"ta_{ta.replace(' ', '_')}"] = int(area == ta)

            rows.append(rec)

    return pd.DataFrame(rows)


def build_xph_processed(data_dir: Path) -> pd.DataFrame:
    xph = pd.read_csv(data_dir / "xph_benchmark.csv")
    xph = xph.rename(
        columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "volume",
        }
    )
    xph["date"] = pd.to_datetime(xph["date"])
    xph["ticker"] = "XPH"
    return xph.sort_values("date").reset_index(drop=True)


def split_train_test(unified: pd.DataFrame):
    train = unified[(unified["date"] >= pd.Timestamp(TRAIN_START)) & (unified["date"] <= pd.Timestamp(TRAIN_END))]
    test = unified[(unified["date"] >= pd.Timestamp(TEST_START)) & (unified["date"] <= pd.Timestamp(TEST_END))]
    return train.copy(), test.copy()


## Clinical Trial Integration (`ct_*`) — Added in Phase 1

Your frustration is valid: some old notebook function cells still show legacy code.

**What actually runs now:**
- The run cell imports and executes `run_pipeline` from `phase1_data_pipeline.py`.
- That `.py` pipeline is the synced source-of-truth and includes clinical-trial merging.

### Clinical trial input used
- `clinical_trials/clinical_trial_event_calendar_fda_matched.csv`

### `ct_*` features written to train/test/unified
- `ct_phase2_events_last_5d`
- `ct_phase3_events_last_5d`
- `ct_results_posted_last_5d`
- `ct_terminated_last_20d`
- `ct_events_last_20d`

### Feature config update
- `processed/feature_config.json` includes `clinical_trial_features`.

If outputs still look old, run the **Run Pipeline** cell and reopen files in `processed/`.

In [52]:
def to_feature_config(unified: pd.DataFrame, out_dir: Path, half_life: float) -> None:
    price_features = [
        "open", "high", "low", "close", "volume", "rsi", "macd", "macd_signal", "macd_diff",
        "bb_upper", "bb_middle", "bb_lower", "bb_pct", "bb_width", "log_return", "volume_sma20", "volume_ratio",
    ]
    sentiment_features = ["sent_pos", "sent_neg", "sent_neu", "sent_net", "n_filings"]
    fda_features = [
        c for c in unified.columns
        if c in {"days_to_event", "is_event_window", "hist_1d_ret", "hist_5d_ret"}
        or c.startswith("evt_")
        or c.startswith("ta_")
    ]

    payload = {
        "tickers": MODEL_TICKERS,
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "test_start": TEST_START,
        "test_end": TEST_END,
        "sentiment_half_life_days": half_life,
        "price_features": price_features,
        "sentiment_features": sentiment_features,
        "fda_features": sorted(fda_features),
        "target_column": "close",
    }
    (out_dir / "feature_config.json").write_text(json.dumps(payload, indent=2))


def run_pipeline(config: Config) -> None:
    data_dir = ensure_data_dir(config)
    config.out_dir.mkdir(parents=True, exist_ok=True)

    price = load_price(data_dir)
    price = price[price["ticker"].isin(MODEL_TICKERS)].copy()
    validate_price_alignment(price)

    sentiment = load_sentiment(data_dir)
    sentiment = sentiment[sentiment["ticker"].isin(MODEL_TICKERS)].copy()
    sentiment_enh = apply_sentiment_decay(sentiment, config.sentiment_half_life)

    fda = load_fda_events(data_dir)
    fda = fda[fda["ticker"].isin(MODEL_TICKERS)].copy()
    fda_daily = build_fda_daily_features(price, fda)

    unified = price.merge(sentiment_enh, on=["date", "ticker"], how="left")
    unified = unified.merge(fda_daily, on=["date", "ticker"], how="left")

    if "days_to_event" in unified.columns:
        unified["days_to_event"] = unified["days_to_event"].fillna(999)
    for col in unified.columns:
        if col in {"date", "ticker", "days_to_event"}:
            continue
        if pd.api.types.is_numeric_dtype(unified[col]):
            unified[col] = unified[col].fillna(0.0)

    unified = unified.sort_values(["date", "ticker"]).reset_index(drop=True)
    train, test = split_train_test(unified)
    xph = build_xph_processed(data_dir)

    unified.to_csv(config.out_dir / "unified_dataset.csv", index=False)
    train.to_csv(config.out_dir / "train_dataset.csv", index=False)
    test.to_csv(config.out_dir / "test_dataset.csv", index=False)
    xph.to_csv(config.out_dir / "xph_processed.csv", index=False)
    to_feature_config(unified, config.out_dir, config.sentiment_half_life)

    print("\nPipeline complete.")
    print(f"data_dir: {data_dir}")
    print(f"out_dir:  {config.out_dir}")
    print(f"unified rows: {len(unified):,}")
    print(f"train rows:   {len(train):,}")
    print(f"test rows:    {len(test):,}")
    print(f"xph rows:     {len(xph):,}")


## Run Pipeline

Pick one config block below:
- Local machine (this project folder)
- Google Colab


In [53]:
# Local machine config (this project)
# IMPORTANT: use the synced pipeline implementation from phase1_data_pipeline.py,
# which includes clinical-trial ct_* features and feature_config updates.
from phase1_data_pipeline import Config, run_pipeline

cfg = Config(
    data_dir=Path("All data DL project"),
    out_dir=Path("processed"),
    zip_path=Path("All data DL project.zip"),
    extract_zip=False,
    sentiment_half_life=5.0,
    clinical_trial_events_path=Path("clinical_trials/clinical_trial_event_calendar_fda_matched.csv"),
)

run_pipeline(cfg)


Date alignment OK: each ticker has 1476 rows.
Loaded clinical trial events from clinical_trials/clinical_trial_event_calendar_fda_matched.csv (4,771 rows).

Pipeline complete.
data_dir: All data DL project
out_dir:  processed
unified rows: 11,808
train rows:   7,800
test rows:    4,008
xph rows:     501


In [54]:
train_df = pd.read_csv("processed/train_dataset.csv")
test_df = pd.read_csv("processed/test_dataset.csv")





In [55]:
train_df.head()


,date,ticker,open,high,low,close,volume,rsi,macd,macd_signal,...,ta_Immunology,ta_Infectious_Disease,ta_Neurology,ta_Oncology,ta_Psychiatry,ct_phase2_events_last_5d,ct_phase3_events_last_5d,ct_results_posted_last_5d,ct_terminated_last_20d,ct_events_last_20d
0,2018-02-20,ABBV,83.545336,86.263471,83.190488,83.729858,8751400,62.590533,2.149340,2.071430,...,0,0,0,0,0,27.0,36.0,0.0,0.0,65.0
1,2018-02-20,AMGN,143.436010,144.549679,142.847804,143.506592,4078400,53.605747,-0.494641,-0.933710,...,0,0,0,0,0,40.0,23.0,9.0,0.0,71.0
2,2018-02-20,BIIB,292.000000,294.880005,286.510010,287.450012,2004500,25.130906,-12.421759,-7.580855,...,0,0,1,0,0,1.0,2.0,0.0,0.0,10.0
3,2018-02-20,BMY,51.219471,51.473740,50.135108,50.366936,8703700,65.657205,0.986764,0.525849,...,0,0,0,1,0,217.0,77.0,13.0,4.0,357.0
4,2018-02-20,GILD,59.041515,60.160370,59.041515,59.578861,7229300,51.331742,0.357360,0.550668,...,1,0,0,0,0,11.0,8.0,0.0,0.0,19.0


In [56]:
test_df.head()

,date,ticker,open,high,low,close,volume,rsi,macd,macd_signal,...,ta_Immunology,ta_Infectious_Disease,ta_Neurology,ta_Oncology,ta_Psychiatry,ct_phase2_events_last_5d,ct_phase3_events_last_5d,ct_results_posted_last_5d,ct_terminated_last_20d,ct_events_last_20d
0,2022-01-03,ABBV,115.838664,116.086743,114.213270,115.847214,6839800,77.240057,3.843451,3.694076,...,1,0,0,0,0,0.0,0.0,0.0,0.0,2.0
1,2022-01-03,AMGN,195.609021,198.530514,194.209497,198.285599,2742800,68.525020,4.238668,3.790104,...,0,0,0,1,0,2.0,0.0,0.0,0.0,3.0
2,2022-01-03,BIIB,240.149994,247.509995,238.070007,244.139999,1642200,52.089344,-1.476641,-3.846023,...,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0
3,2022-01-03,BMY,52.095046,52.237728,51.482361,51.935581,12359400,61.683473,1.001859,0.901914,...,0,0,0,1,0,4.0,0.0,1.0,0.0,14.0
4,2022-01-03,GILD,61.843097,62.202506,61.278323,62.108376,6622500,63.499312,1.025108,1.033998,...,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0


In [57]:
# Colab config (uncomment this block in Colab)
# from phase1_data_pipeline import Config, run_pipeline
# cfg = Config(
#     data_dir=Path("/content/pharma_data/"),
#     out_dir=Path("/content/processed/"),
#     zip_path=Path("/content/pharma_data.zip"),
#     extract_zip=True,
#     sentiment_half_life=5.0,
#     clinical_trial_events_path=Path("/content/clinical_trials/clinical_trial_event_calendar_fda_matched.csv"),
# )
# run_pipeline(cfg)


In [58]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7800 entries, 0 to 7799
Data columns (total 44 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       7800 non-null   object 
 1   ticker                     7800 non-null   object 
 2   open                       7800 non-null   float64
 3   high                       7800 non-null   float64
 4   low                        7800 non-null   float64
 5   close                      7800 non-null   float64
 6   volume                     7800 non-null   int64  
 7   rsi                        7800 non-null   float64
 8   macd                       7800 non-null   float64
 9   macd_signal                7800 non-null   float64
 10  macd_diff                  7800 non-null   float64
 11  bb_upper                   7800 non-null   float64
 12  bb_middle                  7800 non-null   float64
 13  bb_lower                   7800 non-null   float

In [59]:
unified_df = pd.read_csv("processed/unified_dataset.csv")
unified_df.info()






<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11808 entries, 0 to 11807
Data columns (total 44 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       11808 non-null  object 
 1   ticker                     11808 non-null  object 
 2   open                       11808 non-null  float64
 3   high                       11808 non-null  float64
 4   low                        11808 non-null  float64
 5   close                      11808 non-null  float64
 6   volume                     11808 non-null  int64  
 7   rsi                        11808 non-null  float64
 8   macd                       11808 non-null  float64
 9   macd_signal                11808 non-null  float64
 10  macd_diff                  11808 non-null  float64
 11  bb_upper                   11808 non-null  float64
 12  bb_middle                  11808 non-null  float64
 13  bb_lower                   11808 non-null  flo

In [60]:
# Sanity assertion: fail fast if ct_* columns are missing.
required_ct = {
    "ct_phase2_events_last_5d",
    "ct_phase3_events_last_5d",
    "ct_results_posted_last_5d",
    "ct_terminated_last_20d",
    "ct_events_last_20d",
}

train_df = pd.read_csv("processed/train_dataset.csv")
missing = sorted(list(required_ct - set(train_df.columns)))
assert not missing, f"Missing ct columns in train_dataset.csv: {missing}"
print("All required ct_* columns present in train_dataset.csv")

All required ct_* columns present in train_dataset.csv


## Why `xph_processed.csv` Exists

`xph_processed.csv` is **not** used for model training. It is a benchmark series for evaluation.

- The RL agent is trained only on the 8 pharma tickers in `train_dataset.csv`.
- `xph_processed.csv` contains normalized OHLCV for the XPH Pharma ETF.
- In backtesting/evaluation, we compare agent performance vs XPH to report relative metrics such as cumulative return and alpha vs benchmark.

So, XPH is a **reference baseline**, not part of the action space.

In [61]:
train_df.head()

,date,ticker,open,high,low,close,volume,rsi,macd,macd_signal,...,ta_Immunology,ta_Infectious_Disease,ta_Neurology,ta_Oncology,ta_Psychiatry,ct_phase2_events_last_5d,ct_phase3_events_last_5d,ct_results_posted_last_5d,ct_terminated_last_20d,ct_events_last_20d
0,2018-02-20,ABBV,83.545336,86.263471,83.190488,83.729858,8751400,62.590533,2.149340,2.071430,...,0,0,0,0,0,27.0,36.0,0.0,0.0,65.0
1,2018-02-20,AMGN,143.436010,144.549679,142.847804,143.506592,4078400,53.605747,-0.494641,-0.933710,...,0,0,0,0,0,40.0,23.0,9.0,0.0,71.0
2,2018-02-20,BIIB,292.000000,294.880005,286.510010,287.450012,2004500,25.130906,-12.421759,-7.580855,...,0,0,1,0,0,1.0,2.0,0.0,0.0,10.0
3,2018-02-20,BMY,51.219471,51.473740,50.135108,50.366936,8703700,65.657205,0.986764,0.525849,...,0,0,0,1,0,217.0,77.0,13.0,4.0,357.0
4,2018-02-20,GILD,59.041515,60.160370,59.041515,59.578861,7229300,51.331742,0.357360,0.550668,...,1,0,0,0,0,11.0,8.0,0.0,0.0,19.0
